In [1]:
# download_rest.py
import pandas as pd, yfinance as yf, os
import pandas as pd, numpy as np, os, re, datetime, glob


RAW = r"D:\MBFM_project\mbfm dataset"
OUT = r"D:\MBFM_project"


In [3]:

OUT = r"D:\MBFM_project\mbfm dataset"
os.makedirs(OUT, exist_ok=True)
S, E = "2005-01-01", "2026-10-12"

for name, sid in {"vix": "VIXCLS", "brent": "DCOILBRENTEU", "us10y": "DGS10"}.items():
    df = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}")
    df.columns = ["date", name]
    df = df[(df.date >= S) & (df.date <= E)]
    df.to_csv(os.path.join(OUT, f"{name}_fred_raw.csv"), index=False)
    print(name, df.shape)

for name, tk in {"dxy": "DX-Y.NYB", "eem": "EEM", "nifty": "^NSEI", "usdinr_yf": "INR=X"}.items():
    df = yf.download(tk, start=S, end=E, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df[["Close"]].rename(columns={"Close": name}).to_csv(os.path.join(OUT, f"{name}_yahoo_raw.csv"))
    print(name, df.shape)

vix (5681, 2)
brent (5681, 2)
us10y (5681, 2)
dxy (5483, 6)
eem (5477, 6)
nifty (4677, 6)
usdinr_yf (5645, 6)


In [4]:
import pandas as pd, glob, os
D = r"D:\MBFM_project\mbfm dataset"
files = glob.glob(os.path.join(D, "*_fred_raw.csv")) + glob.glob(os.path.join(D, "*_yahoo_raw.csv"))
for f in sorted(files):
    df = pd.read_csv(f)
    d = pd.to_datetime(df.iloc[:, 0])
    v = pd.to_numeric(df.iloc[:, 1], errors="coerce")
    print(f"{os.path.basename(f):22s} {d.min().date()} -> {d.max().date()}  rows={len(df)}  NaN={v.isna().sum()}")

brent_fred_raw.csv     2005-01-03 -> 2026-10-12  rows=5681  NaN=232
dxy_yahoo_raw.csv      2005-01-03 -> 2026-10-09  rows=5483  NaN=0
eem_yahoo_raw.csv      2005-01-03 -> 2026-10-09  rows=5477  NaN=0
nifty_yahoo_raw.csv    2007-09-17 -> 2026-10-09  rows=4677  NaN=0
us10y_fred_raw.csv     2005-01-03 -> 2026-10-12  rows=5681  NaN=233
usdinr_yf_yahoo_raw.csv 2005-01-03 -> 2026-10-09  rows=5645  NaN=0
vix_fred_raw.csv       2005-01-03 -> 2026-10-12  rows=5681  NaN=170


In [6]:
D = r"D:\MBFM_project\mbfm dataset"
rd = lambda p, **k: pd.read_csv(p, **k) if p.lower().endswith((".csv", ".txt")) else pd.read_excel(p, **k)

usd = glob.glob(os.path.join(D, "Daily Exchange Rate*"))[0]
cm  = glob.glob(os.path.join(D, "Daily Weighted*"))[0]
i10 = os.path.join(D, "india10y_raw.csv")

for name, p, kw in [("USDINR RBI", usd, {"header": None}),
                    ("CALLMONEY", cm, {"header": None}),
                    ("INDIA10Y", i10, {})]:
    df = rd(p, **kw)
    print(f"--- {name} {df.shape} ---")
    print(df.head(6).to_string())
    print(df.tail(3).to_string())

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


--- USDINR RBI (5273, 3) ---
    0                                        1          2
0 NaN                                      NaN        NaN
1 NaN  Daily Exchange Rate of the Indian Rupee        NaN
2 NaN                                      NaN        NaN
3 NaN        ( ₹ per unit of foreign currency)        NaN
4 NaN                                      NaN        NaN
5 NaN                                     Date  US Dollar
       0                                                                                                                                                                                                                                                           1      2
5270 NaN                                                                                                                                                                                                                                         2005-01-03 00:00:00  43.61
5271 NaN                     

In [7]:
import pandas as pd, numpy as np, os, re, datetime, glob
D = r"D:\MBFM_project\mbfm dataset"
OUT = r"D:\MBFM_project"

def rd(p, **k):
    return pd.read_csv(p, **k) if p.lower().endswith((".csv", ".txt")) else pd.read_excel(p, **k)

def to_date(x):
    if isinstance(x, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(x)
    if isinstance(x, str) and re.fullmatch(r"\d{1,2}-[A-Za-z]{3}-\d{2,4}", x.strip()):
        return pd.to_datetime(x.strip(), format="mixed", errors="coerce")
    return pd.NaT

def build(dates, vals):
    s = pd.DataFrame({"d": [to_date(x) for x in dates], "v": pd.to_numeric(vals, errors="coerce")}).dropna()
    return s[s.d >= "2005-01-01"].drop_duplicates("d").set_index("d")["v"].sort_index()

def simple(p):
    df = pd.read_csv(p)
    return build(pd.to_datetime(df.iloc[:, 0], errors="coerce"), df.iloc[:, 1])

# RBI USD/INR (anchor calendar), call money (borrowing weighted rate), India 10Y
df = rd(glob.glob(os.path.join(D, "Daily Exchange Rate*"))[0], header=None)
usd = build(df[1], df[2])
df = rd(glob.glob(os.path.join(D, "Daily Weighted*"))[0], header=None)
cm = build(df[1], df[4])
df = pd.read_csv(os.path.join(D, "india10y_raw.csv"))
i10 = build(pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce"),
            df["Price"].astype(str).str.replace(",", ""))

S = {n: simple(os.path.join(D, f"{n}_fred_raw.csv")) for n in ["vix", "brent", "us10y"]}
S.update({n: simple(os.path.join(D, f"{n}_yahoo_raw.csv")) for n in ["dxy", "eem", "nifty"]})

# optional Nifty 2005-2007 splice (Investing.com export)
p = os.path.join(D, "nifty_2005_2007_raw.csv")
if os.path.exists(p):
    e = pd.read_csv(p)
    e = build(pd.to_datetime(e["Date"], format="%d-%m-%Y", errors="coerce"),
              e["Price"].astype(str).str.replace(",", ""))
    S["nifty"] = pd.concat([e[e.index < S["nifty"].index.min()], S["nifty"]]).sort_index()
else:
    print("NOTE: no Nifty 2005-2007 file; usable sample will start ~Sept 2007")

ALL = {"usdinr": usd, "callmoney": cm, "india10y": i10, **S}
for n, s in ALL.items():
    print(f"{n:10s} {s.index.min().date()} -> {s.index.max().date()}  n={len(s)}  "
          f"maxgap={s.index.to_series().diff().dt.days.max():.0f}d  range={s.min():.2f}..{s.max():.2f}")

# master calendar = RBI USD/INR dates
A = usd.index
m = pd.DataFrame({"usdinr": usd})
for n, s in {**S, "india10y": i10, "callmoney": cm}.items():
    m[n] = s.reindex(A, method="ffill", limit=5)

r = np.log(m.usdinr).diff()
rv = r.rolling(22).std()          # target: 22-day realized vol, known at close of day t
L = lambda s: s.shift(1)

f = pd.DataFrame(index=A)
f["rv_lag1"]      = L(rv)
f["rv_5d"]        = L(rv).rolling(5).mean()
f["rv_22d"]       = L(rv).rolling(22).mean()
f["sq_return"]    = L(r ** 2)
f["vix_level"]    = L(m.vix);      f["vix_chg"]      = L(m.vix.diff())
f["brent_ret"]    = L(np.log(m.brent).diff())
f["dxy_level"]    = L(m.dxy);      f["dxy_ret"]      = L(np.log(m.dxy).diff())
f["india10y"]     = L(m.india10y); f["india10y_chg"] = L(m.india10y.diff())
f["us10y"]        = L(m.us10y);    f["us10y_chg"]    = L(m.us10y.diff())
f["rate_spread"]  = L(m.india10y - m.us10y)
f["callmoney"]    = L(m.callmoney)
f["nifty_ret"]    = L(np.log(m.nifty).diff())
f["em_ret"]       = L(np.log(m.eem).diff())

# known-in-advance inputs (NOT lagged)
f["day_of_week"]  = A.dayofweek
f["month"]        = A.month
f["week_of_year"] = A.isocalendar().week.to_numpy().astype(int)

def flag(path, next_day=False):
    if not os.path.exists(path):
        print("MISSING:", path, "-> flag set to all zeros"); return 0
    d = pd.to_datetime(pd.read_csv(path)["date"])
    if next_day:                                   # FOMC statement is after Indian close
        pos = A.searchsorted(d, side="right")
        d = A[pos[pos < len(A)]]
    return A.isin(d).astype(int)

f["is_mpc_day"]  = flag(os.path.join(D, "mpc_dates.csv"))
f["is_fomc_day"] = flag(os.path.join(D, "fomc_dates.csv"), next_day=True)

f["ret_t"] = r        # GARCH input only, NOT an ML feature
f["rv"]    = rv       # target

print(f.isna().sum()[lambda s: s > 0])
data = f.dropna()
print(data.shape, data.index.min().date(), data.index.max().date())
print(data[["rv", "callmoney", "india10y", "vix_level", "rate_spread"]].describe().T[["min", "mean", "max"]])
data.to_csv(os.path.join(OUT, "master_features.csv"))

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


NOTE: no Nifty 2005-2007 file; usable sample will start ~Sept 2007
usdinr     2005-01-03 -> 2026-10-12  n=5261  maxgap=6d  range=39.27..96.84
callmoney  2005-04-01 -> 2026-10-21  n=6455  maxgap=10d  range=0.13..54.32
india10y   2005-01-01 -> 2024-12-26  n=5000  maxgap=6d  range=5.17..9.48
vix        2005-01-03 -> 2026-10-12  n=5511  maxgap=5d  range=9.14..82.69
brent      2005-01-04 -> 2026-10-12  n=5449  maxgap=9d  range=9.12..143.95
us10y      2005-01-03 -> 2026-10-09  n=5448  maxgap=4d  range=0.52..5.31
dxy        2005-01-03 -> 2026-10-09  n=5483  maxgap=5d  range=71.33..114.11
eem        2005-01-03 -> 2026-10-09  n=5477  maxgap=5d  range=18.26..71.21
nifty      2007-09-17 -> 2026-10-09  n=4677  maxgap=6d  range=2524.20..26328.55
MISSING: D:\MBFM_project\mbfm dataset\mpc_dates.csv -> flag set to all zeros
MISSING: D:\MBFM_project\mbfm dataset\fomc_dates.csv -> flag set to all zeros
rv_lag1          23
rv_5d            27
rv_22d           44
sq_return         2
vix_level         1
vi

In [1]:
import os
import pandas as pd

DATA_DIR = r"D:\MBFM_project"
RAW_DIR = os.path.join(DATA_DIR, "mbfm dataset")

def check_dates(filepath):
    if not os.path.exists(filepath):
        return "FILE NOT FOUND"
    
    # Read first row to inspect structure
    df_sample = pd.read_csv(filepath, nrows=1)
    
    # Check if 'date' is already a named column or if index col 0 is the date
    if any(c.lower() in ["date", "time"] for c in df_sample.columns):
        date_col = [c for c in df_sample.columns if c.lower() in ["date", "time"]][0]
        df = pd.read_csv(filepath)
        s = df[date_col]
    else:
        # Date is stored in index column 0
        df = pd.read_csv(filepath, index_col=0)
        s = df.index

    parsed_dates = pd.to_datetime(s, format="mixed", dayfirst=True, errors="coerce").dropna()
    return f"Start: {parsed_dates.min().strftime('%Y-%m-%d')} | End: {parsed_dates.max().strftime('%Y-%m-%d')} | Rows: {len(parsed_dates)}"

print("=== RAW DATASETS ===")
for f in sorted(os.listdir(RAW_DIR)):
    if f.endswith(".csv"):
        print(f"{f:<30} -> {check_dates(os.path.join(RAW_DIR, f))}")

print("\n=== COMPILED MASTER FEATURES ===")
master_path = os.path.join(DATA_DIR, "master_features.csv")
print(f"master_features.csv            -> {check_dates(master_path)}")

=== RAW DATASETS ===
brent_fred_raw.csv             -> Start: 2005-01-02 | End: 2026-12-10 | Rows: 5681
dxy_yahoo_raw.csv              -> Start: 2005-01-02 | End: 2026-12-08 | Rows: 5483
eem_yahoo_raw.csv              -> Start: 2005-01-02 | End: 2026-12-08 | Rows: 5477
india10y_2025_2026_raw.csv     -> Start: 2024-02-12 | End: 2026-12-10 | Rows: 485
india10y_raw.csv               -> Start: 2005-01-01 | End: 2024-12-26 | Rows: 5000
nifty_2005_2007_raw.csv        -> Start: 2005-01-02 | End: 2007-12-31 | Rows: 750
nifty_yahoo_raw.csv            -> Start: 2007-01-10 | End: 2026-12-08 | Rows: 4677
us10y_fred_raw.csv             -> Start: 2005-01-02 | End: 2026-12-10 | Rows: 5681
usdinr_yf_yahoo_raw.csv        -> Start: 2005-01-02 | End: 2026-12-08 | Rows: 5645
vix_fred_raw.csv               -> Start: 2005-01-02 | End: 2026-12-10 | Rows: 5681

=== COMPILED MASTER FEATURES ===
master_features.csv            -> Start: 2007-01-10 | End: 2025-03-01 | Rows: 4170


In [3]:
import pandas as pd, numpy as np, os, re, datetime, glob

RAW = r"D:\MBFM_project\mbfm dataset"
OUT = r"D:\MBFM_project"

# ── helpers ────────────────────────────────────────────────────────
def to_ts(x):
    if pd.isnull(x): return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime.datetime)): return pd.Timestamp(x)
    s = str(x).strip()
    for fmt in ("%d-%m-%Y", "%d-%b-%Y", "%d-%b-%y", "%Y-%m-%d", "%m/%d/%Y"):
        try: return pd.to_datetime(s, format=fmt)
        except: pass
    return pd.to_datetime(s, errors="coerce")

def build(dates, vals):
    s = pd.DataFrame({"d": [to_ts(x) for x in dates],
                       "v": pd.to_numeric(vals, errors="coerce")}).dropna()
    s = s[s.d >= "2005-01-01"].drop_duplicates("d").set_index("d")["v"].sort_index()
    return s

def simple_csv(p):
    df = pd.read_csv(p)
    return build(pd.to_datetime(df.iloc[:,0], errors="coerce"), df.iloc[:,1])

# ── India 10Y: merge old (2005-2024) + new (2024-2026) ────────────
df_old = pd.read_csv(os.path.join(RAW, "india10y_raw.csv"))
i10_old = build(pd.to_datetime(df_old["Date"], format="%d-%m-%Y", errors="coerce"),
                df_old["Price"].astype(str).str.replace(",",""))

df_new = pd.read_csv(os.path.join(RAW, "india10y_2025_2026_raw.csv"))
i10_new = build(pd.to_datetime(df_new["Date"], format="%d-%m-%Y", errors="coerce"),
                df_new["Price"].astype(str).str.replace(",",""))

i10 = pd.concat([i10_old, i10_new]).sort_index()
i10 = i10[~i10.index.duplicated(keep="last")]
print(f"India 10Y merged: {i10.index.min().date()} -> {i10.index.max().date()}  n={len(i10)}")

# ── Nifty: merge 2005-2007 + Yahoo ────────────────────────────────
nifty_yf  = simple_csv(os.path.join(RAW, "nifty_yahoo_raw.csv"))

df_n07 = pd.read_csv(os.path.join(RAW, "nifty_2005_2007_raw.csv"))
nifty_07 = build(pd.to_datetime(df_n07["Date"], format="%d-%m-%Y", errors="coerce"),
                 df_n07["Price"].astype(str).str.replace(",",""))

nifty = pd.concat([nifty_07[nifty_07.index < nifty_yf.index.min()], nifty_yf]).sort_index()
nifty = nifty[~nifty.index.duplicated(keep="last")]
print(f"Nifty merged:     {nifty.index.min().date()} -> {nifty.index.max().date()}  n={len(nifty)}")

# ── other series ──────────────────────────────────────────────────
vix   = simple_csv(os.path.join(RAW, "vix_fred_raw.csv"))
brent = simple_csv(os.path.join(RAW, "brent_fred_raw.csv"))
us10y = simple_csv(os.path.join(RAW, "us10y_fred_raw.csv"))
dxy   = simple_csv(os.path.join(RAW, "dxy_yahoo_raw.csv"))
eem   = simple_csv(os.path.join(RAW, "eem_yahoo_raw.csv"))

# RBI USD/INR — anchor calendar
df_fx = pd.read_excel(glob.glob(os.path.join(RAW, "Daily Exchange Rate*xlsx"))[0], header=None)
usdinr = build(df_fx[1], df_fx[2])
print(f"USD/INR anchor:   {usdinr.index.min().date()} -> {usdinr.index.max().date()}  n={len(usdinr)}")

# Call money
df_cm = pd.read_excel(glob.glob(os.path.join(RAW, "Daily Weighted*xlsx"))[0], header=None)
cm = build(df_cm[1], df_cm[4])
print(f"Call money:       {cm.index.min().date()} -> {cm.index.max().date()}  n={len(cm)}")

# ── align to anchor calendar ──────────────────────────────────────
A = usdinr.index
m = pd.DataFrame({"usdinr": usdinr})
for name, s in {"vix":vix,"brent":brent,"us10y":us10y,"dxy":dxy,
                 "eem":eem,"nifty":nifty,"india10y":i10,"callmoney":cm}.items():
    m[name] = s.reindex(A, method="ffill", limit=5)

print("\nNaN counts after alignment:")
print(m.isna().sum()[m.isna().sum()>0])

# ── features ──────────────────────────────────────────────────────
r  = np.log(m.usdinr).diff()
rv = r.rolling(22).std()
L  = lambda s: s.shift(1)

f = pd.DataFrame(index=A)
f["rv_lag1"]      = L(rv)
f["rv_5d"]        = L(rv).rolling(5).mean()
f["rv_22d"]       = L(rv).rolling(22).mean()
f["sq_return"]    = L(r**2)
f["vix_level"]    = L(m.vix);       f["vix_chg"]       = L(m.vix.diff())
f["brent_ret"]    = L(np.log(m.brent).diff())
f["dxy_level"]    = L(m.dxy);       f["dxy_ret"]       = L(np.log(m.dxy).diff())
f["india10y"]     = L(m.india10y);  f["india10y_chg"]  = L(m.india10y.diff())
f["us10y"]        = L(m.us10y);     f["us10y_chg"]     = L(m.us10y.diff())
f["rate_spread"]  = L(m.india10y - m.us10y)
f["callmoney"]    = L(m.callmoney)
f["nifty_ret"]    = L(np.log(m.nifty).diff())
f["em_ret"]       = L(np.log(m.eem).diff())
f["day_of_week"]  = A.dayofweek
f["month"]        = A.month
f["week_of_year"] = A.isocalendar().week.to_numpy().astype(int)

# MPC flag
mpc_path = os.path.join(RAW, "mpc_dates.csv")
if os.path.exists(mpc_path):
    mpc_dates = pd.to_datetime(pd.read_csv(mpc_path)["date"])
    f["is_mpc_day"] = A.isin(mpc_dates).astype(int)
    print(f"MPC flag: {f.is_mpc_day.sum()} meeting days found")
else:
    f["is_mpc_day"] = 0
    print("WARNING: mpc_dates.csv missing — flag set to 0")

# FOMC flag (shifted to next Indian trading day)
fomc_path = os.path.join(RAW, "fomc_dates.csv")
if os.path.exists(fomc_path):
    fomc_dates = pd.to_datetime(pd.read_csv(fomc_path)["date"])
    pos = A.searchsorted(fomc_dates, side="right")
    fomc_ind = A[pos[pos < len(A)]]
    f["is_fomc_day"] = A.isin(fomc_ind).astype(int)
    print(f"FOMC flag: {f.is_fomc_day.sum()} meeting days found")
else:
    f["is_fomc_day"] = 0
    print("WARNING: fomc_dates.csv missing — flag set to 0")

f["ret_t"] = r
f["rv"]    = rv

# ── final drop NaN ────────────────────────────────────────────────
data = f.dropna()
print(f"\nmaster_features: {data.shape}")
print(f"Start: {data.index.min().date()}  End: {data.index.max().date()}")
print(data[["rv","callmoney","india10y","vix_level","rate_spread"]].describe()
          .T[["min","mean","max"]].round(4))

data.to_csv(os.path.join(OUT, "master_features.csv"))
print("\nSaved master_features.csv")

India 10Y merged: 2005-01-01 -> 2024-12-26  n=5000
Nifty merged:     2007-09-17 -> 2026-10-09  n=4677


d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


USD/INR anchor:   2005-01-03 -> 2026-10-12  n=5261


d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Call money:       2005-04-01 -> 2026-10-21  n=6455

NaN counts after alignment:
brent          2
nifty        661
india10y     428
callmoney     61
dtype: int64

master_features: (4170, 24)
Start: 2007-09-19  End: 2025-01-03
                min     mean      max
rv           0.0003   0.0038   0.0177
callmoney    2.9900   6.1747  19.7000
india10y     5.1720   7.4159   9.4780
vix_level    9.1400  19.9266  82.6900
rate_spread  2.1640   4.7728   7.0610

Saved master_features.csv


In [6]:
#---- helpers---------------

def to_ts(x):
    if pd.isnull(x): return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime.datetime)): return pd.Timestamp(x)
    s = str(x).strip()
    for fmt in ("%d-%m-%Y", "%d-%b-%Y", "%d-%b-%y", "%Y-%m-%d", "%m/%d/%Y"):
        try: return pd.to_datetime(s, format=fmt)
        except: pass
    return pd.to_datetime(s, errors="coerce")

def build(dates, vals):
    s = pd.DataFrame({"d": [to_ts(x) for x in dates],
                       "v": pd.to_numeric(vals, errors="coerce")}).dropna()
    s = s[s.d >= "2005-01-01"].drop_duplicates("d").set_index("d")["v"].sort_index()
    return s

def simple_csv(p):
    df = pd.read_csv(p)
    return build(pd.to_datetime(df.iloc[:,0], errors="coerce"), df.iloc[:,1])

# ── Nifty: merge 2005-2007 + 2008-2026 (Both Investing.com) ─────────
df_n07 = pd.read_csv(os.path.join(RAW, "nifty_2005_2007_raw.csv"))
nifty_07 = build(df_n07["Date"], df_n07["Price"].astype(str).str.replace(",",""))

df_n08 = pd.read_csv(os.path.join(RAW, "nifty_2008_2026_raw.csv"))
nifty_08 = build(df_n08["Date"], df_n08["Price"].astype(str).str.replace(",",""))

# Clean non-overlapping splice
nifty = pd.concat([nifty_07, nifty_08[nifty_08.index > nifty_07.index.max()]]).sort_index()
nifty = nifty[~nifty.index.duplicated(keep="last")]
print(f"Nifty merged:     {nifty.index.min().date()} -> {nifty.index.max().date()}  n={len(nifty)}")

Nifty merged:     2005-01-03 -> 2026-10-27  n=5410


In [7]:
# ── Nifty: merge 2005-2007 + 2008-2026 (Both Investing.com) ─────────
df_n07 = pd.read_csv(os.path.join(RAW, "nifty_2005_2007_raw.csv"))
nifty_07 = build(df_n07["Date"], df_n07["Price"].astype(str).str.replace(",",""))

df_n08 = pd.read_csv(os.path.join(RAW, "nifty_2008_2026_raw.csv"))
nifty_08 = build(df_n08["Date"], df_n08["Price"].astype(str).str.replace(",",""))

# Clean non-overlapping splice
nifty = pd.concat([nifty_07, nifty_08[nifty_08.index > nifty_07.index.max()]]).sort_index()
nifty = nifty[~nifty.index.duplicated(keep="last")]
print(f"Nifty merged:     {nifty.index.min().date()} -> {nifty.index.max().date()}  n={len(nifty)}")

Nifty merged:     2005-01-03 -> 2026-10-27  n=5410


In [8]:
# ── Export Complete Nifty Dataset (All Columns) ─────────────────────
df_n07 = pd.read_csv(os.path.join(RAW, "nifty_2005_2007_raw.csv"))
df_n08 = pd.read_csv(os.path.join(RAW, "nifty_2008_2026_raw.csv"))

# Parse dates and sort
df_n07["date_parsed"] = pd.to_datetime(df_n07["Date"], format="mixed", dayfirst=False, errors="coerce")
df_n08["date_parsed"] = pd.to_datetime(df_n08["Date"], format="mixed", dayfirst=False, errors="coerce")

df_n07 = df_n07.dropna(subset=["date_parsed"]).sort_values("date_parsed")
df_n08 = df_n08.dropna(subset=["date_parsed"]).sort_values("date_parsed")

# Merge non-overlapping full dataframes
df_n08_clean = df_n08[df_n08["date_parsed"] > df_n07["date_parsed"].max()]
nifty_full_df = pd.concat([df_n07, df_n08_clean]).drop(columns=["date_parsed"])

nifty_out_path = os.path.join(RAW, "nifty_merged.csv")
nifty_full_df.to_csv(nifty_out_path, index=False)
print(f"Saved complete Nifty dataset with all columns to: {nifty_out_path}")

Saved complete Nifty dataset with all columns to: D:\MBFM_project\mbfm dataset\nifty_merged.csv


In [9]:
import os
import pandas as pd

RAW = r"D:\MBFM_project\mbfm dataset"

# 1. Load both raw files
df_old = pd.read_csv(os.path.join(RAW, "india10y_raw.csv"))
df_new = pd.read_csv(os.path.join(RAW, "india10y_2025_2026_raw.csv"))

# 2. Parse dates safely to handle DD-MM-YYYY vs MM/DD/YYYY
df_old["_date_parsed"] = pd.to_datetime(df_old["Date"], format="mixed", dayfirst=True, errors="coerce")
df_new["_date_parsed"] = pd.to_datetime(df_new["Date"], format="mixed", dayfirst=True, errors="coerce")

df_old = df_old.dropna(subset=["_date_parsed"]).sort_values("_date_parsed")
df_new = df_new.dropna(subset=["_date_parsed"]).sort_values("_date_parsed")

# 3. Non-overlapping splice: append new data strictly after old max date
df_new_clean = df_new[df_new["_date_parsed"] > df_old["_date_parsed"].max()]
india10y_merged = pd.concat([df_old, df_new_clean]).drop(columns=["_date_parsed"])

# 4. Save merged CSV keeping all columns intact
out_path = os.path.join(RAW, "india10y_merged.csv")
india10y_merged.to_csv(out_path, index=False)

print("=== India 10Y Merge Successful ===")
print(f"Total Rows   : {len(india10y_merged)}")
print(f"Columns Saved: {list(india10y_merged.columns)}")
print(f"Saved to     : {out_path}")

=== India 10Y Merge Successful ===
Total Rows   : 5467
Columns Saved: ['Date', 'Price', 'Open', 'High', 'Low', 'Change %']
Saved to     : D:\MBFM_project\mbfm dataset\india10y_merged.csv


In [10]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

RAW = r"D:\MBFM_project\mbfm dataset"
os.makedirs(RAW, exist_ok=True)

# ── 1. Scrape RBI MPC Announcement Dates ─────────────────────────────
print("Scraping RBI MPC dates...")
rbi_url = "https://www.rbi.org.in/scripts/Annualpolicy.aspx"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

mpc_dates = []
try:
    res = requests.get(rbi_url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")
    # Find all 'Bi-monthly Monetary Policy Statement' sections
    for table in soup.find_all("table"):
        text = table.get_text()
        matches = re.findall(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{1,2}),\s+(\d{4})", text)
        for m in matches:
            dt_str = f"{m[0]} {m[1]}, {m[2]}"
            dt = pd.to_datetime(dt_str, format="%b %d, %Y", errors="coerce")
            if pd.notnull(dt) and dt.year >= 2016:
                mpc_dates.append(dt.strftime("%Y-%m-%d"))
except Exception as e:
    print(f"Scraper notice: {e}")

# Static official RBI MPC backup (Oct 2016 - Oct 2026) to ensure 100% complete coverage
mpc_backup = [
    "2016-10-04", "2016-12-07", "2017-02-08", "2017-04-06", "2017-06-07", "2017-08-02", "2017-10-04", "2017-12-06",
    "2018-02-07", "2018-04-05", "2018-06-06", "2018-08-01", "2018-10-05", "2018-12-05", "2019-02-07", "2019-04-04",
    "2019-06-06", "2019-08-07", "2019-10-04", "2019-12-05", "2020-03-27", "2020-05-22", "2020-08-06", "2020-10-09",
    "2020-12-04", "2021-02-05", "2021-04-07", "2021-06-04", "2021-08-06", "2021-10-08", "2021-12-08", "2022-02-10",
    "2022-05-04", "2022-06-08", "2022-08-05", "2022-09-30", "2022-12-07", "2023-02-08", "2023-04-06", "2023-06-08",
    "2023-08-10", "2023-10-06", "2023-12-08", "2024-02-08", "2024-04-05", "2024-06-07", "2024-08-08", "2024-10-09",
    "2024-12-06", "2025-02-07", "2025-04-09", "2025-06-06", "2025-08-08", "2025-10-08", "2025-12-05", "2026-02-06",
    "2026-04-08", "2026-06-05", "2026-08-05", "2026-10-07"
]

all_mpc = sorted(list(set(mpc_dates + mpc_backup)))
pd.DataFrame({"date": all_mpc}).to_csv(os.path.join(RAW, "mpc_dates.csv"), index=False)
print(f"Saved mpc_dates.csv ({len(all_mpc)} dates)")

# ── 2. Scrape/Generate US FOMC Statement Dates ──────────────────────
print("Generating FOMC dates...")
fomc_backup = [
    # 2005-2009
    "2005-02-02", "2005-03-16", "2005-05-03", "2005-06-30", "2005-08-09", "2005-09-20", "2005-11-01", "2005-12-13",
    "2006-01-31", "2006-03-28", "2006-05-10", "2006-06-29", "2006-08-08", "2006-09-20", "2006-10-25", "2006-12-12",
    "2007-01-31", "2007-03-21", "2007-05-09", "2007-06-28", "2007-08-07", "2007-09-18", "2007-10-31", "2007-12-11",
    "2008-01-30", "2008-03-18", "2008-04-30", "2008-06-25", "2008-08-05", "2008-09-16", "2008-10-29", "2008-12-16",
    "2009-01-28", "2009-03-18", "2009-04-29", "2009-06-24", "2009-08-12", "2009-09-23", "2009-11-04", "2009-12-16",
    # 2010-2019
    "2010-01-27", "2010-03-16", "2010-04-28", "2010-06-23", "2010-08-10", "2010-09-21", "2010-11-03", "2010-12-14",
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22", "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20", "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19", "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18", "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17", "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15", "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14", "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13", "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19", "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020-2026
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29", "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05", "2020-12-16",
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16", "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15", "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14", "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12", "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18", "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
    "2026-01-28", "2026-03-18", "2026-04-29", "2026-06-17", "2026-07-29", "2026-09-16", "2026-10-28"
]

all_fomc = sorted(list(set(fomc_backup)))
pd.DataFrame({"date": all_fomc}).to_csv(os.path.join(RAW, "fomc_dates.csv"), index=False)
print(f"Saved fomc_dates.csv ({len(all_fomc)} dates)")

Scraping RBI MPC dates...
Saved mpc_dates.csv (69 dates)
Generating FOMC dates...
Saved fomc_dates.csv (176 dates)


In [11]:
import os
import pandas as pd

RAW = r"D:\MBFM_project\mbfm dataset"
mpc_path = os.path.join(RAW, "mpc_dates.csv")

# Exact official decision announcement dates (Oct 2016 - Oct 2026)
official_mpc_decisions = [
    # 2016 - 2018
    "2016-10-04", "2016-12-07", "2017-02-08", "2017-04-06", "2017-06-07", "2017-08-02", "2017-10-04", "2017-12-06",
    "2018-02-07", "2018-04-05", "2018-06-06", "2018-08-01", "2018-10-05", "2018-12-05",
    # 2019 - 2021
    "2019-02-07", "2019-04-04", "2019-06-06", "2019-08-07", "2019-10-04", "2019-12-05",
    "2020-03-27", "2020-05-22", "2020-08-06", "2020-10-09", "2020-12-04",
    "2021-02-05", "2021-04-07", "2021-06-04", "2021-08-06", "2021-10-08", "2021-12-08",
    # 2022 - 2024
    "2022-02-10", "2022-05-04", "2022-06-08", "2022-08-05", "2022-09-30", "2022-12-07",
    "2023-02-08", "2023-04-06", "2023-06-08", "2023-08-10", "2023-10-06", "2023-12-08",
    "2024-02-08", "2024-04-05", "2024-06-07", "2024-08-08", "2024-10-09", "2024-12-06",
    # 2025 - 2026
    "2025-02-07", "2025-04-09", "2025-06-06", "2025-08-08", "2025-10-08", "2025-12-05",
    "2026-02-06", "2026-04-08", "2026-06-05", "2026-08-05", "2026-10-07"
]

pd.DataFrame({"date": sorted(official_mpc_decisions)}).to_csv(mpc_path, index=False)
print(f"Cleaned mpc_dates.csv: kept exactly {len(official_mpc_decisions)} decision announcement days.")

Cleaned mpc_dates.csv: kept exactly 60 decision announcement days.


In [12]:
import os
import pandas as pd

DATA_DIR = r"D:\MBFM_project"
RAW_DIR = os.path.join(DATA_DIR, "mbfm dataset")

# 1. Load Master Features & CSV Date Lists
master_df = pd.read_csv(os.path.join(DATA_DIR, "master_features.csv"), parse_dates=["date"]).set_index("date")

mpc_csv_dates = pd.to_datetime(pd.read_csv(os.path.join(RAW_DIR, "mpc_dates.csv"))["date"]).dt.strftime('%Y-%m-%d').tolist()
fomc_csv_dates = pd.to_datetime(pd.read_csv(os.path.join(RAW_DIR, "fomc_dates.csv"))["date"]).dt.strftime('%Y-%m-%d').tolist()

# 2. Extract Flagged Dates from Master Features
mpc_flagged = master_df[master_df["is_mpc_day"] == 1].index.strftime('%Y-%m-%d').tolist()
fomc_flagged = master_df[master_df["is_fomc_day"] == 1].index.strftime('%Y-%m-%d').tolist()

print("==================================================")
print(" 1. RBI MPC ANNOUNCEMENT DATES CHECK")
print("==================================================")
print(f"Total dates in mpc_dates.csv       : {len(mpc_csv_dates)}")
print(f"Total dates flagged in Master DF  : {len(mpc_flagged)}")

# Check if any CSV dates fell on non-trading days/weekends
unmatched_mpc = [d for d in mpc_csv_dates if d not in mpc_flagged]
if unmatched_mpc:
    print(f"Notice: {len(unmatched_mpc)} MPC dates fell on non-trading days/holidays:")
    print(" ", unmatched_mpc)
else:
    print("All MPC dates perfectly matched the trading calendar!")

print("\nRecent 5 Flagged MPC Days in Dataset:")
print(mpc_flagged[-5:])


print("\n==================================================")
print(" 2. US FOMC STATEMENT DATES CHECK")
print("==================================================")
print(f"Total dates in fomc_dates.csv      : {len(fomc_csv_dates)}")
print(f"Total dates flagged in Master DF  : {len(fomc_flagged)}")

# FOMC is intentionally shifted to the NEXT Indian trading day
print("\nRecent 5 FOMC Statement Dates vs Flagged Indian Trading Days:")
for f_date in fomc_csv_dates[-5:]:
    pos = master_df.index.searchsorted(pd.Timestamp(f_date), side="right")
    if pos < len(master_df.index):
        mapped_day = master_df.index[pos].strftime('%Y-%m-%d')
        print(f"  US Statement: {f_date}  -->  Flagged Indian Day (T+1): {mapped_day}")

print("==================================================")

ValueError: Missing column provided to 'parse_dates': 'date'

In [13]:
import os
import pandas as pd

RAW = r"D:\MBFM_project\mbfm dataset"
fomc_path = os.path.join(RAW, "fomc_dates.csv")

# Exact FOMC Statement Release Days (2005 - 2026)
official_fomc_statements = [
    # 2005
    "2005-02-02", "2005-03-16", "2005-05-03", "2005-06-30", "2005-08-09", "2005-09-20", "2005-11-01", "2005-12-13",
    # 2006
    "2006-01-31", "2006-03-28", "2006-05-10", "2006-06-29", "2006-08-08", "2006-09-20", "2006-10-25", "2006-12-12",
    # 2007
    "2007-01-31", "2007-03-21", "2007-05-09", "2007-06-28", "2007-08-07", "2007-09-18", "2007-10-31", "2007-12-11",
    # 2008 (includes Jan 22 & Oct 8 inter-meeting cuts)
    "2008-01-22", "2008-01-30", "2008-03-18", "2008-04-30", "2008-06-25", "2008-08-05", "2008-09-16", "2008-10-08", "2008-10-29", "2008-12-16",
    # 2009
    "2009-01-28", "2009-03-18", "2009-04-29", "2009-06-24", "2009-08-12", "2009-09-23", "2009-11-04", "2009-12-16",
    # 2010
    "2010-01-27", "2010-03-16", "2010-04-28", "2010-06-23", "2010-08-10", "2010-09-21", "2010-11-03", "2010-12-14",
    # 2011
    "2011-01-26", "2011-03-15", "2011-04-27", "2011-06-22", "2011-08-09", "2011-09-21", "2011-11-02", "2011-12-13",
    # 2012
    "2012-01-25", "2012-03-13", "2012-04-25", "2012-06-20", "2012-08-01", "2012-09-13", "2012-10-24", "2012-12-12",
    # 2013
    "2013-01-30", "2013-03-20", "2013-05-01", "2013-06-19", "2013-07-31", "2013-09-18", "2013-10-30", "2013-12-18",
    # 2014
    "2014-01-29", "2014-03-19", "2014-04-30", "2014-06-18", "2014-07-30", "2014-09-17", "2014-10-29", "2014-12-17",
    # 2015
    "2015-01-28", "2015-03-18", "2015-04-29", "2015-06-17", "2015-07-29", "2015-09-17", "2015-10-28", "2015-12-16",
    # 2016
    "2016-01-27", "2016-03-16", "2016-04-27", "2016-06-15", "2016-07-27", "2016-09-21", "2016-11-02", "2016-12-14",
    # 2017
    "2017-02-01", "2017-03-15", "2017-05-03", "2017-06-14", "2017-07-26", "2017-09-20", "2017-11-01", "2017-12-13",
    # 2018
    "2018-01-31", "2018-03-21", "2018-05-02", "2018-06-13", "2018-08-01", "2018-09-26", "2018-11-08", "2018-12-19",
    # 2019
    "2019-01-30", "2019-03-20", "2019-05-01", "2019-06-19", "2019-07-31", "2019-09-18", "2019-10-30", "2019-12-11",
    # 2020 (includes Mar 3 & Mar 15 emergency cuts)
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29", "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05", "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16", "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15", "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14", "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12", "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07", "2025-06-18", "2025-07-30", "2025-09-17", "2025-10-29", "2025-12-10",
    # 2026
    "2026-01-28", "2026-03-18", "2026-04-29", "2026-06-17", "2026-07-29", "2026-09-16", "2026-10-28"
]

pd.DataFrame({"date": sorted(official_fomc_statements)}).to_csv(fomc_path, index=False)
print(f"Cleaned fomc_dates.csv: kept exactly {len(official_fomc_statements)} decision statement days.")

Cleaned fomc_dates.csv: kept exactly 178 decision statement days.


### making the master features csv file

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
base = Path(r"D:\MBFM_project\mbfm dataset")
out  = Path(r"D:\MBFM_project\master_features.csv")

# ── 1. USD/INR (anchor calendar) ──────────────────────────────────────────
usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()          # col 1 = date, col 2 = USD
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

cal = usdinr[["date"]].copy()

# ── 2. Helper: align any series to calendar ────────────────────────────────
def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

# ── 3. Load each series ────────────────────────────────────────────────────

# Call money — col 1 = date, col 4 = borrowing weighted rate
cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

# India 10Y (merged, DD-MM-YYYY)
i10y = pd.read_csv(base / "india10y_merged.csv")
india10y_s = align(i10y, "Date", "Price", "india10y", dayfirst=True)

# FRED series (date, value)
vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

# Yahoo (Close column)
dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),      "Date", "dxy",      "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),       "Date", "eem",      "em_close")
nifty_s     = align(pd.read_csv(base / "nifty_merged.csv"),        "Date", "Price",    "nifty_close")
usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"),"Date", "usdinr_yf","usdinr_yf")

# ── 4. Assemble base frame ─────────────────────────────────────────────────
df = cal.copy().set_index("date")
df["usdinr"]    = usdinr.set_index("date")["usdinr"]
df["callmoney"] = callmoney_s
df["india10y"]  = india10y_s
df["vix_level"] = vix_s
df["brent"]     = brent_s
df["us10y"]     = us10y_s
df["dxy_level"] = dxy_s
df["em_close"]  = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"] = usdinr_yf_s

df = df.sort_index()

# ── 5. Returns & changes ───────────────────────────────────────────────────
df["ret_t"]        = np.log(df["usdinr"]).diff()
df["vix_chg"]      = df["vix_level"].diff()
df["brent_ret"]    = np.log(df["brent"]).diff()
df["dxy_ret"]      = np.log(df["dxy_level"]).diff()
df["india10y_chg"] = df["india10y"].diff()
df["us10y_chg"]    = df["us10y"].diff()
df["rate_spread"]  = df["india10y"] - df["us10y"]
df["nifty_ret"]    = np.log(df["nifty_close"]).diff()
df["em_ret"]       = np.log(df["em_close"]).diff()

# ── 6. Realized Volatility (22-day rolling std of log returns) ─────────────
df["rv"] = df["ret_t"].rolling(22).std() * np.sqrt(252)   # annualised

# ── 7. Lagged RV features ──────────────────────────────────────────────────
df["rv_lag1"] = df["rv"].shift(1)
df["rv_5d"]   = df["rv"].shift(1).rolling(5).mean()
df["rv_22d"]  = df["rv"].shift(1).rolling(22).mean()
df["sq_return"] = df["ret_t"].shift(1) ** 2

# ── 8. Calendar dummies ────────────────────────────────────────────────────
df["day_of_week"]  = df.index.dayofweek          # 0=Mon
df["month"]        = df.index.month
df["week_of_year"] = df.index.isocalendar().week.astype(int)

# ── 9. MPC flag (unlagged — announcement day itself) ──────────────────────
mpc = pd.read_csv(base / "mpc_dates.csv")
mpc["date"] = pd.to_datetime(mpc["date"], errors="coerce")
mpc_set = set(mpc["date"].dropna())
df["is_mpc_day"] = df.index.isin(mpc_set).astype(int)

# ── 10. FOMC flag (shifted to next Indian trading day) ────────────────────
fomc = pd.read_csv(base / "fomc_dates.csv")
fomc["date"] = pd.to_datetime(fomc["date"], errors="coerce")
fomc_dates = fomc["date"].dropna().sort_values()

trading_days = df.index.sort_values()
def next_trading_day(d):
    future = trading_days[trading_days >= d]
    return future[0] if len(future) else pd.NaT

fomc_shifted = pd.Series([next_trading_day(d) for d in fomc_dates]).dropna()
df["is_fomc_day"] = df.index.isin(set(fomc_shifted)).astype(int)

# ── 11. Drop rows with NaN in any feature (burn-in) ───────────────────────
feature_cols = [
    "rv", "rv_lag1", "rv_5d", "rv_22d", "sq_return",
    "vix_level", "vix_chg", "brent_ret", "dxy_level", "dxy_ret",
    "india10y", "india10y_chg", "us10y", "us10y_chg",
    "rate_spread", "callmoney", "nifty_ret", "em_ret",
    "ret_t", "is_mpc_day", "is_fomc_day",
    "day_of_week", "month", "week_of_year"
]
df = df[feature_cols].dropna()

# ── 12. Save ───────────────────────────────────────────────────────────────
df.to_csv(out)
print(f"Saved: {out}")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"\nColumn check ({len(df.columns)} cols):\n{list(df.columns)}")
print(f"\nNull check:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nSanity checks:")
print(f"  usdinr range (from ret_t):  should be ~43-96")
print(f"  callmoney range:            {df['callmoney'].min():.2f} – {df['callmoney'].max():.2f}  (expect ~3-9)")
print(f"  india10y max daily gap:     check manually if worried")
print(f"  is_mpc_day sum:             {df['is_mpc_day'].sum()}  (expect ~60-80 post Oct-2016)")
print(f"  is_fomc_day sum:            {df['is_fomc_day'].sum()}  (expect ~170)")
print(f"  rv head:\n{df['rv'].head(30).to_string()}")

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Saved: D:\MBFM_project\master_features.csv
Shape: (0, 24)
Date range: NaT → NaT

Column check (24 cols):
['rv', 'rv_lag1', 'rv_5d', 'rv_22d', 'sq_return', 'vix_level', 'vix_chg', 'brent_ret', 'dxy_level', 'dxy_ret', 'india10y', 'india10y_chg', 'us10y', 'us10y_chg', 'rate_spread', 'callmoney', 'nifty_ret', 'em_ret', 'ret_t', 'is_mpc_day', 'is_fomc_day', 'day_of_week', 'month', 'week_of_year']

Null check:
Series([], dtype: int64)

Sanity checks:
  usdinr range (from ret_t):  should be ~43-96
  callmoney range:            nan – nan  (expect ~3-9)
  india10y max daily gap:     check manually if worried
  is_mpc_day sum:             0  (expect ~60-80 post Oct-2016)
  is_fomc_day sum:            0  (expect ~170)
  rv head:
Series([], )


In [5]:
for fname in ["vix_fred_raw.csv", "brent_fred_raw.csv", "us10y_fred_raw.csv"]:
    tmp = pd.read_csv(base / fname, nrows=2)
    print(fname, "→", list(tmp.columns))

vix_fred_raw.csv → ['date', 'vix']
brent_fred_raw.csv → ['date', 'brent']
us10y_fred_raw.csv → ['date', 'us10y']


In [4]:
for fname in ["dxy_yahoo_raw.csv", "eem_yahoo_raw.csv", "nifty_merged.csv", "usdinr_yf_yahoo_raw.csv"]:
    tmp = pd.read_csv(base / fname, nrows=2)
    print(fname, "→", list(tmp.columns))

dxy_yahoo_raw.csv → ['Date', 'dxy']
eem_yahoo_raw.csv → ['Date', 'eem']
nifty_merged.csv → ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
usdinr_yf_yahoo_raw.csv → ['Date', 'usdinr_yf']


In [6]:
# Check actual layout of both Excel files
for fname in ["Daily Exchange Rate of the Indian Rupee.xlsx",
              "Daily Weighted Average Call_Notice Money Rates.xlsx"]:
    tmp = pd.read_excel(base / fname, header=None, nrows=15)
    print(f"\n{'='*60}")
    print(f"{fname}")
    print(tmp.to_string())


Daily Exchange Rate of the Indian Rupee.xlsx
     0                                        1          2
0  NaN                                      NaN        NaN
1  NaN  Daily Exchange Rate of the Indian Rupee        NaN
2  NaN                                      NaN        NaN
3  NaN        ( ₹ per unit of foreign currency)        NaN
4  NaN                                      NaN        NaN
5  NaN                                     Date  US Dollar
6  NaN                                        1          2
7  NaN                      2026-10-12 00:00:00    95.7485
8  NaN                      2026-10-09 00:00:00    96.6149
9  NaN                      2026-10-08 00:00:00    96.7733
10 NaN                      2026-10-07 00:00:00     96.623
11 NaN                      2026-10-06 00:00:00    96.4348
12 NaN                      2026-10-05 00:00:00    96.2831
13 NaN                      2026-10-01 00:00:00    95.9927
14 NaN                      2026-09-30 00:00:00    95.9832

Daily Wei

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
base = Path(r"D:\MBFM_project\mbfm dataset")
out  = Path(r"D:\MBFM_project\master_features.csv")

# ── 1. USD/INR (anchor calendar) ──────────────────────────────────────────
usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()          # col 1 = date, col 2 = USD
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

cal = usdinr[["date"]].copy()

# ── 2. Helper: align any series to calendar ────────────────────────────────
def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

# ── 3. Load each series ────────────────────────────────────────────────────

# Call money — col 1 = date, col 4 = borrowing weighted rate
cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

# India 10Y (merged, DD-MM-YYYY)
i10y = pd.read_csv(base / "india10y_merged.csv")
india10y_s = align(i10y, "Date", "Price", "india10y", dayfirst=True)

# FRED series (date, value)
vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

# Yahoo (Close column)
dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),      "Date", "dxy",      "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),       "Date", "eem",      "em_close")
nifty_s     = align(pd.read_csv(base / "nifty_merged.csv"),        "Date", "Price",    "nifty_close")
usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"),"Date", "usdinr_yf","usdinr_yf")

# ── 4. Assemble base frame ─────────────────────────────────────────────────
df = cal.copy().set_index("date")
df["usdinr"]    = usdinr.set_index("date")["usdinr"]
df["callmoney"] = callmoney_s
df["india10y"]  = india10y_s
df["vix_level"] = vix_s
df["brent"]     = brent_s
df["us10y"]     = us10y_s
df["dxy_level"] = dxy_s
df["em_close"]  = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"] = usdinr_yf_s

df = df.sort_index()

# ── 5. Returns & changes ───────────────────────────────────────────────────
df["ret_t"]        = np.log(df["usdinr"]).diff()
df["vix_chg"]      = df["vix_level"].diff()
df["brent_ret"]    = np.log(df["brent"]).diff()
df["dxy_ret"]      = np.log(df["dxy_level"]).diff()
df["india10y_chg"] = df["india10y"].diff()
df["us10y_chg"]    = df["us10y"].diff()
df["rate_spread"]  = df["india10y"] - df["us10y"]
df["nifty_ret"]    = np.log(df["nifty_close"]).diff()
df["em_ret"]       = np.log(df["em_close"]).diff()

# ── 6. Realized Volatility (22-day rolling std of log returns) ─────────────
df["rv"] = df["ret_t"].rolling(22).std() * np.sqrt(252)   # annualised

# ── 7. Lagged RV features ──────────────────────────────────────────────────
df["rv_lag1"] = df["rv"].shift(1)
df["rv_5d"]   = df["rv"].shift(1).rolling(5).mean()
df["rv_22d"]  = df["rv"].shift(1).rolling(22).mean()
df["sq_return"] = df["ret_t"].shift(1) ** 2

# ── 8. Calendar dummies ────────────────────────────────────────────────────
df["day_of_week"]  = df.index.dayofweek          # 0=Mon
df["month"]        = df.index.month
df["week_of_year"] = df.index.isocalendar().week.astype(int)

# ── 9. MPC flag (unlagged — announcement day itself) ──────────────────────
mpc = pd.read_csv(base / "mpc_dates.csv")
mpc["date"] = pd.to_datetime(mpc["date"], errors="coerce")
mpc_set = set(mpc["date"].dropna())
df["is_mpc_day"] = df.index.isin(mpc_set).astype(int)

# ── 10. FOMC flag (shifted to next Indian trading day) ────────────────────
fomc = pd.read_csv(base / "fomc_dates.csv")
fomc["date"] = pd.to_datetime(fomc["date"], errors="coerce")
fomc_dates = fomc["date"].dropna().sort_values()

trading_days = df.index.sort_values()
def next_trading_day(d):
    future = trading_days[trading_days >= d]
    return future[0] if len(future) else pd.NaT

fomc_shifted = pd.Series([next_trading_day(d) for d in fomc_dates]).dropna()
df["is_fomc_day"] = df.index.isin(set(fomc_shifted)).astype(int)

# ── 11. Drop rows with NaN in any feature (burn-in) ───────────────────────
feature_cols = [
    "rv", "rv_lag1", "rv_5d", "rv_22d", "sq_return",
    "vix_level", "vix_chg", "brent_ret", "dxy_level", "dxy_ret",
    "india10y", "india10y_chg", "us10y", "us10y_chg",
    "rate_spread", "callmoney", "nifty_ret", "em_ret",
    "ret_t", "is_mpc_day", "is_fomc_day",
    "day_of_week", "month", "week_of_year"
]
df = df[feature_cols].dropna()

# ── 12. Save ───────────────────────────────────────────────────────────────
df.to_csv(out)
print(f"Saved: {out}")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"\nColumn check ({len(df.columns)} cols):\n{list(df.columns)}")
print(f"\nNull check:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nSanity checks:")
print(f"  usdinr range (from ret_t):  should be ~43-96")
print(f"  callmoney range:            {df['callmoney'].min():.2f} – {df['callmoney'].max():.2f}  (expect ~3-9)")
print(f"  india10y max daily gap:     check manually if worried")
print(f"  is_mpc_day sum:             {df['is_mpc_day'].sum()}  (expect ~60-80 post Oct-2016)")
print(f"  is_fomc_day sum:            {df['is_fomc_day'].sum()}  (expect ~170)")
print(f"  rv head:\n{df['rv'].head(30).to_string()}")

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\robin\AppData\Local\Temp\ipykernel_20892\3231231874.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")


Saved: D:\MBFM_project\master_features.csv
Shape: (0, 24)
Date range: NaT → NaT

Column check (24 cols):
['rv', 'rv_lag1', 'rv_5d', 'rv_22d', 'sq_return', 'vix_level', 'vix_chg', 'brent_ret', 'dxy_level', 'dxy_ret', 'india10y', 'india10y_chg', 'us10y', 'us10y_chg', 'rate_spread', 'callmoney', 'nifty_ret', 'em_ret', 'ret_t', 'is_mpc_day', 'is_fomc_day', 'day_of_week', 'month', 'week_of_year']

Null check:
Series([], dtype: int64)

Sanity checks:
  usdinr range (from ret_t):  should be ~43-96
  callmoney range:            nan – nan  (expect ~3-9)
  india10y max daily gap:     check manually if worried
  is_mpc_day sum:             0  (expect ~60-80 post Oct-2016)
  is_fomc_day sum:            0  (expect ~170)
  rv head:
Series([], )


In [8]:
# Diagnose BEFORE dropna
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(r"D:\MBFM_project\mbfm dataset")

usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

print("USD/INR shape:", usdinr.shape)
print("USD/INR date range:", usdinr["date"].min(), "→", usdinr["date"].max())
print("USD/INR sample:\n", usdinr.head(3).to_string())

cal = usdinr[["date"]].copy()

def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

i10y = pd.read_csv(base / "india10y_merged.csv")
india10y_s = align(i10y, "Date", "Price", "india10y", dayfirst=True)

vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),       "Date", "dxy",       "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),        "Date", "eem",       "em_close")
nifty_s     = align(pd.read_csv(base / "nifty_merged.csv"),         "Date", "Price",     "nifty_close")
usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"), "Date", "usdinr_yf", "usdinr_yf")

df = cal.copy().set_index("date")
df["usdinr"]      = usdinr.set_index("date")["usdinr"]
df["callmoney"]   = callmoney_s
df["india10y"]    = india10y_s
df["vix_level"]   = vix_s
df["brent"]       = brent_s
df["us10y"]       = us10y_s
df["dxy_level"]   = dxy_s
df["em_close"]    = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"]   = usdinr_yf_s
df = df.sort_index()

print("\ndf shape before any dropna:", df.shape)
print("NaN counts per column:")
print(df.isnull().sum().to_string())
print("\nFirst 3 rows:")
print(df.head(3).to_string())
print("\nLast 3 rows:")
print(df.tail(3).to_string())

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


USD/INR shape: (5260, 2)
USD/INR date range: 2005-01-03 00:00:00 → 2026-10-12 00:00:00
USD/INR sample:
         date  usdinr
0 2005-01-03   43.61
1 2005-01-04   43.43
2 2005-01-05   43.77


d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



df shape before any dropna: (5260, 10)
NaN counts per column:
usdinr            0
callmoney        61
india10y        428
vix_level         0
brent             2
us10y             0
dxy_level         0
em_close          0
nifty_close    5260
usdinr_yf        10

First 3 rows:
            usdinr  callmoney  india10y  vix_level  brent  us10y  dxy_level   em_close  nifty_close  usdinr_yf
date                                                                                                          
2005-01-03   43.61        NaN     6.538      14.08    NaN   4.23  81.300003  22.194445          NaN  43.416000
2005-01-04   43.43        NaN     6.485      13.98  40.75   4.29  82.570000  21.511110          NaN  43.491001
2005-01-05   43.77        NaN     6.567      14.09  41.00   4.29  82.540001  21.247778          NaN  43.589001

Last 3 rows:
             usdinr  callmoney  india10y  vix_level   brent  us10y   dxy_level   em_close  nifty_close  usdinr_yf
date                                   

C:\Users\robin\AppData\Local\Temp\ipykernel_20892\1243407101.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")


In [9]:
nifty = pd.read_csv(base / "nifty_merged.csv")
print("nifty_merged columns:", list(nifty.columns))
print(nifty.head(3).to_string())
print(nifty.tail(3).to_string())
print("shape:", nifty.shape)

print()
i10y = pd.read_csv(base / "india10y_merged.csv")
print("india10y_merged columns:", list(i10y.columns))
print(i10y.head(3).to_string())
print(i10y.tail(3).to_string())
print("shape:", i10y.shape)

nifty_merged columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
         Date     Price      Open      High       Low     Vol. Change %
0  01/03/2005  2,115.00  2,080.00  2,118.60  2,080.00   70.51M    1.66%
1  01/04/2005  2,103.75  2,116.95  2,120.15  2,100.55   72.72M   -0.53%
2  01/05/2005  2,032.20  2,103.75  2,105.10  1,990.15  109.22M   -3.40%
            Date      Price       Open       High        Low     Vol. Change %
5407  10/23/2026  22,600.00  22,742.20  22,787.25  22,372.00  364.19M   -0.39%
5408  10/26/2026  22,753.20  22,524.40  22,753.20  22,387.35  310.22M    0.68%
5409  10/27/2026  22,749.75  22,708.80  22,826.85  22,638.75  400.95M   -0.02%
shape: (5410, 7)

india10y_merged columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Change %']
         Date  Price   Open   High    Low Change %
0  01-01-2005  6.795  6.795  6.795  6.795    1.33%
1  03-01-2005  6.538  6.538  6.538  6.538   -3.78%
2  04-01-2005  6.485  6.485  6.485  6.485   -0.81%
            D

In [10]:
# Nifty — strip commas from Price before passing to align
nifty_raw = pd.read_csv(base / "nifty_merged.csv")
nifty_raw["Price"] = nifty_raw["Price"].astype(str).str.replace(",", "").str.strip()
nifty_s = align(nifty_raw, "Date", "Price", "nifty_close", dayfirst=True)

# India 10Y — force consistent date parsing
i10y_raw = pd.read_csv(base / "india10y_merged.csv")
i10y_raw["Date"] = i10y_raw["Date"].astype(str).str.replace("/", "-")
india10y_s = align(i10y_raw, "Date", "Price", "india10y", dayfirst=True)

In [11]:
# Diagnose BEFORE dropna
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(r"D:\MBFM_project\mbfm dataset")

usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

print("USD/INR shape:", usdinr.shape)
print("USD/INR date range:", usdinr["date"].min(), "→", usdinr["date"].max())
print("USD/INR sample:\n", usdinr.head(3).to_string())

cal = usdinr[["date"]].copy()

def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

i10y = pd.read_csv(base / "india10y_merged.csv")
india10y_s = align(i10y, "Date", "Price", "india10y", dayfirst=True)

vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),       "Date", "dxy",       "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),        "Date", "eem",       "em_close")
nifty_s     = align(pd.read_csv(base / "nifty_merged.csv"),         "Date", "Price",     "nifty_close")
usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"), "Date", "usdinr_yf", "usdinr_yf")

df = cal.copy().set_index("date")
df["usdinr"]      = usdinr.set_index("date")["usdinr"]
df["callmoney"]   = callmoney_s
df["india10y"]    = india10y_s
df["vix_level"]   = vix_s
df["brent"]       = brent_s
df["us10y"]       = us10y_s
df["dxy_level"]   = dxy_s
df["em_close"]    = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"]   = usdinr_yf_s
df = df.sort_index()

print("\ndf shape before any dropna:", df.shape)
print("NaN counts per column:")
print(df.isnull().sum().to_string())
print("\nFirst 3 rows:")
print(df.head(3).to_string())
print("\nLast 3 rows:")
print(df.tail(3).to_string())

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


USD/INR shape: (5260, 2)
USD/INR date range: 2005-01-03 00:00:00 → 2026-10-12 00:00:00
USD/INR sample:
         date  usdinr
0 2005-01-03   43.61
1 2005-01-04   43.43
2 2005-01-05   43.77


d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\robin\AppData\Local\Temp\ipykernel_20892\1243407101.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")



df shape before any dropna: (5260, 10)
NaN counts per column:
usdinr            0
callmoney        61
india10y        428
vix_level         0
brent             2
us10y             0
dxy_level         0
em_close          0
nifty_close    5260
usdinr_yf        10

First 3 rows:
            usdinr  callmoney  india10y  vix_level  brent  us10y  dxy_level   em_close  nifty_close  usdinr_yf
date                                                                                                          
2005-01-03   43.61        NaN     6.538      14.08    NaN   4.23  81.300003  22.194445          NaN  43.416000
2005-01-04   43.43        NaN     6.485      13.98  40.75   4.29  82.570000  21.511110          NaN  43.491001
2005-01-05   43.77        NaN     6.567      14.09  41.00   4.29  82.540001  21.247778          NaN  43.589001

Last 3 rows:
             usdinr  callmoney  india10y  vix_level   brent  us10y   dxy_level   em_close  nifty_close  usdinr_yf
date                                   

In [12]:
nifty_raw = pd.read_csv(base / "nifty_merged.csv")
nifty_raw["Price"] = nifty_raw["Price"].astype(str).str.replace(",", "").str.strip()
print("Nifty Price sample:", nifty_raw["Price"].head(3).tolist())
print("Nifty Date sample:", nifty_raw["Date"].head(3).tolist())

i10y_raw = pd.read_csv(base / "india10y_merged.csv")
i10y_raw["Date"] = i10y_raw["Date"].astype(str).str.replace("/", "-")
print("\ni10y Date sample:", i10y_raw["Date"].head(3).tolist())
print("i10y Date tail:", i10y_raw["Date"].tail(3).tolist())

Nifty Price sample: ['2115.00', '2103.75', '2032.20']
Nifty Date sample: ['01/03/2005', '01/04/2005', '01/05/2005']

i10y Date sample: ['01-01-2005', '03-01-2005', '04-01-2005']
i10y Date tail: ['06-12-2026', '08-12-2026', '10-12-2026']


In [18]:
# Diagnose BEFORE dropna
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(r"D:\MBFM_project\mbfm dataset")

usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

print("USD/INR shape:", usdinr.shape)
print("USD/INR date range:", usdinr["date"].min(), "→", usdinr["date"].max())
print("USD/INR sample:\n", usdinr.head(3).to_string())

cal = usdinr[["date"]].copy()

def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

# India 10Y — normalize date separators

i10y_raw = pd.read_csv(base / "india10y_merged.csv")

def parse_india10y_date(d):
    d = str(d).strip()
    if "/" in d:
        return pd.to_datetime(d, format="%m/%d/%Y")  # new part: MM/DD/YYYY
    else:
        return pd.to_datetime(d, format="%d-%m-%Y")  # old part: DD-MM-YYYY

i10y_raw["Date"] = i10y_raw["Date"].apply(parse_india10y_date)
india10y_s = align(i10y_raw, "Date", "Price", "india10y", dayfirst=False)

vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),       "Date", "dxy",       "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),        "Date", "eem",       "em_close")

# Nifty — strip commas from Price

nifty_raw = pd.read_csv(base / "nifty_merged.csv")
nifty_raw["Price"] = nifty_raw["Price"].astype(str).str.replace(",", "").str.strip()
nifty_raw["Date"] = pd.to_datetime(nifty_raw["Date"], format="%m/%d/%Y")
nifty_s = align(nifty_raw, "Date", "Price", "nifty_close", dayfirst=False)

usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"), "Date", "usdinr_yf", "usdinr_yf")

df = cal.copy().set_index("date")
df["usdinr"]      = usdinr.set_index("date")["usdinr"]
df["callmoney"]   = callmoney_s
df["india10y"]    = india10y_s
df["vix_level"]   = vix_s
df["brent"]       = brent_s
df["us10y"]       = us10y_s
df["dxy_level"]   = dxy_s
df["em_close"]    = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"]   = usdinr_yf_s
df = df.sort_index()

print("\ndf shape before any dropna:", df.shape)
print("NaN counts per column:")
print(df.isnull().sum().to_string())
print("\nFirst 3 rows:")
print(df.head(3).to_string())
print("\nLast 3 rows:")
print(df.tail(3).to_string())

# Extra: check nifty and india10y specifically
print("\nNifty sample (first non-NaN):")
print(df["nifty_close"].dropna().head(3))
print("\nIndia10y tail (last 5):")
print(df["india10y"].tail(5))

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


USD/INR shape: (5260, 2)
USD/INR date range: 2005-01-03 00:00:00 → 2026-10-12 00:00:00
USD/INR sample:
         date  usdinr
0 2005-01-03   43.61
1 2005-01-04   43.43
2 2005-01-05   43.77


d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\robin\AppData\Local\Temp\ipykernel_20892\1592268851.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")



df shape before any dropna: (5260, 10)
NaN counts per column:
usdinr          0
callmoney      61
india10y        0
vix_level       0
brent           2
us10y           0
dxy_level       0
em_close        0
nifty_close     0
usdinr_yf      10

First 3 rows:
            usdinr  callmoney  india10y  vix_level  brent  us10y  dxy_level   em_close  nifty_close  usdinr_yf
date                                                                                                          
2005-01-03   43.61        NaN     6.538      14.08    NaN   4.23  81.300003  22.194445      2115.00  43.416000
2005-01-04   43.43        NaN     6.485      13.98  40.75   4.29  82.570000  21.511110      2103.75  43.491001
2005-01-05   43.77        NaN     6.567      14.09  41.00   4.29  82.540001  21.247778      2032.20  43.589001

Last 3 rows:
             usdinr  callmoney  india10y  vix_level   brent  us10y   dxy_level   em_close  nifty_close  usdinr_yf
date                                                       

In [14]:
nifty_raw = pd.read_csv(base / "nifty_merged.csv")
print(nifty_raw["Date"].head(5).tolist())
print(nifty_raw["Date"].tail(5).tolist())

['01/03/2005', '01/04/2005', '01/05/2005', '01/06/2005', '01/07/2005']
['10/21/2026', '10/22/2026', '10/23/2026', '10/26/2026', '10/27/2026']


In [16]:
i10y_raw = pd.read_csv(base / "india10y_merged.csv")
print("First 3:", i10y_raw["Date"].head(3).tolist())
print("Around row 5000:", i10y_raw["Date"].iloc[4998:5003].tolist())
print("Last 3:", i10y_raw["Date"].tail(3).tolist())
print("Total rows:", len(i10y_raw))

First 3: ['01-01-2005', '03-01-2005', '04-01-2005']
Around row 5000: ['24-12-2024', '26-12-2024', '12/27/2024', '12/30/2024', '12/31/2024']
Last 3: ['06/12/2026', '08/12/2026', '10/12/2026']
Total rows: 5467


In [17]:
i10y_raw = pd.read_csv(base / "india10y_merged.csv")

def parse_india10y_date(d):
    d = str(d).strip()
    if "/" in d:
        return pd.to_datetime(d, format="%m/%d/%Y")  # new part: MM/DD/YYYY
    else:
        return pd.to_datetime(d, format="%d-%m-%Y")  # old part: DD-MM-YYYY

i10y_raw["Date"] = i10y_raw["Date"].apply(parse_india10y_date)
india10y_s = align(i10y_raw, "Date", "Price", "india10y", dayfirst=False)

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(r"D:\MBFM_project\mbfm dataset")
out  = Path(r"D:\MBFM_project\master_features.csv")

# ── 1. USD/INR (anchor calendar) ──────────────────────────────────────────
usdinr = pd.read_excel(base / "Daily Exchange Rate of the Indian Rupee.xlsx",
                       header=None, skiprows=7)
usdinr = usdinr.iloc[:, [1, 2]].copy()
usdinr.columns = ["date", "usdinr"]
usdinr = usdinr[pd.to_datetime(usdinr["date"], errors="coerce").notna()].copy()
usdinr["date"] = pd.to_datetime(usdinr["date"])
usdinr["usdinr"] = pd.to_numeric(usdinr["usdinr"], errors="coerce")
usdinr = usdinr.dropna().sort_values("date").reset_index(drop=True)
usdinr = usdinr[usdinr["date"].dt.dayofweek < 5].copy()

cal = usdinr[["date"]].copy()

# ── 2. Helper ──────────────────────────────────────────────────────────────
def align(df, date_col, val_col, new_name, ffill_limit=5, dayfirst=False):
    df = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df[date_col]):
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")
    df = df[[date_col, val_col]].dropna(subset=[date_col])
    df.columns = ["date", new_name]
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date")
    merged = cal.merge(df, on="date", how="left")
    merged[new_name] = merged[new_name].ffill(limit=ffill_limit)
    return merged.set_index("date")[new_name]

# ── 3. Load each series ────────────────────────────────────────────────────
cm_raw = pd.read_excel(base / "Daily Weighted Average Call_Notice Money Rates.xlsx",
                       header=None, skiprows=7)
callmoney_s = align(cm_raw, cm_raw.columns[1], cm_raw.columns[4], "callmoney")

# India 10Y — mixed date formats
i10y_raw = pd.read_csv(base / "india10y_merged.csv")
def parse_india10y_date(d):
    d = str(d).strip()
    if "/" in d:
        return pd.to_datetime(d, format="%m/%d/%Y")
    else:
        return pd.to_datetime(d, format="%d-%m-%Y")
i10y_raw["Date"] = i10y_raw["Date"].apply(parse_india10y_date)
india10y_s = align(i10y_raw, "Date", "Price", "india10y", dayfirst=False)

# FRED
vix_s   = align(pd.read_csv(base / "vix_fred_raw.csv"),   "date", "vix",   "vix_level")
brent_s = align(pd.read_csv(base / "brent_fred_raw.csv"), "date", "brent", "brent")
us10y_s = align(pd.read_csv(base / "us10y_fred_raw.csv"), "date", "us10y", "us10y")

# Yahoo
dxy_s       = align(pd.read_csv(base / "dxy_yahoo_raw.csv"),       "Date", "dxy",       "dxy_level")
eem_s       = align(pd.read_csv(base / "eem_yahoo_raw.csv"),        "Date", "eem",       "em_close")
usdinr_yf_s = align(pd.read_csv(base / "usdinr_yf_yahoo_raw.csv"), "Date", "usdinr_yf", "usdinr_yf")

# Nifty — comma-separated prices, MM/DD/YYYY
nifty_raw = pd.read_csv(base / "nifty_merged.csv")
nifty_raw["Price"] = nifty_raw["Price"].astype(str).str.replace(",", "").str.strip()
nifty_raw["Date"]  = pd.to_datetime(nifty_raw["Date"], format="%m/%d/%Y")
nifty_s = align(nifty_raw, "Date", "Price", "nifty_close", dayfirst=False)

# ── 4. Assemble ────────────────────────────────────────────────────────────
df = cal.copy().set_index("date")
df["usdinr"]      = usdinr.set_index("date")["usdinr"]
df["callmoney"]   = callmoney_s
df["india10y"]    = india10y_s
df["vix_level"]   = vix_s
df["brent"]       = brent_s
df["us10y"]       = us10y_s
df["dxy_level"]   = dxy_s
df["em_close"]    = eem_s
df["nifty_close"] = nifty_s
df["usdinr_yf"]   = usdinr_yf_s
df = df.sort_index()

# ── 5. Returns & changes ───────────────────────────────────────────────────
df["ret_t"]        = np.log(df["usdinr"]).diff()
df["vix_chg"]      = df["vix_level"].diff()
df["brent_ret"]    = np.log(df["brent"]).diff()
df["dxy_ret"]      = np.log(df["dxy_level"]).diff()
df["india10y_chg"] = df["india10y"].diff()
df["us10y_chg"]    = df["us10y"].diff()
df["rate_spread"]  = df["india10y"] - df["us10y"]
df["nifty_ret"]    = np.log(df["nifty_close"]).diff()
df["em_ret"]       = np.log(df["em_close"]).diff()

# ── 6. Realized Volatility ─────────────────────────────────────────────────
df["rv"] = df["ret_t"].rolling(22).std() * np.sqrt(252)

# ── 7. Lagged RV features ──────────────────────────────────────────────────
df["rv_lag1"]   = df["rv"].shift(1)
df["rv_5d"]     = df["rv"].shift(1).rolling(5).mean()
df["rv_22d"]    = df["rv"].shift(1).rolling(22).mean()
df["sq_return"] = df["ret_t"].shift(1) ** 2

# ── 8. Calendar ────────────────────────────────────────────────────────────
df["day_of_week"]  = df.index.dayofweek
df["month"]        = df.index.month
df["week_of_year"] = df.index.isocalendar().week.astype(int)

# ── 9. MPC flag ────────────────────────────────────────────────────────────
mpc = pd.read_csv(base / "mpc_dates.csv")
mpc["date"] = pd.to_datetime(mpc["date"], errors="coerce")
df["is_mpc_day"] = df.index.isin(set(mpc["date"].dropna())).astype(int)

# ── 10. FOMC flag (next Indian trading day) ───────────────────────────────
fomc = pd.read_csv(base / "fomc_dates.csv")
fomc["date"] = pd.to_datetime(fomc["date"], errors="coerce")
fomc_dates = fomc["date"].dropna().sort_values()
trading_days = df.index.sort_values()

def next_trading_day(d):
    future = trading_days[trading_days >= d]
    return future[0] if len(future) else pd.NaT

fomc_shifted = pd.Series([next_trading_day(d) for d in fomc_dates]).dropna()
df["is_fomc_day"] = df.index.isin(set(fomc_shifted)).astype(int)

# ── 11. Select & drop burn-in ──────────────────────────────────────────────
feature_cols = [
    "rv", "rv_lag1", "rv_5d", "rv_22d", "sq_return",
    "vix_level", "vix_chg", "brent_ret", "dxy_level", "dxy_ret",
    "india10y", "india10y_chg", "us10y", "us10y_chg",
    "rate_spread", "callmoney", "nifty_ret", "em_ret",
    "ret_t", "is_mpc_day", "is_fomc_day",
    "day_of_week", "month", "week_of_year"
]
df = df[feature_cols].dropna()

# ── 12. Save & report ──────────────────────────────────────────────────────
df.to_csv(out)
print(f"Saved: {out}")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"\nColumns ({len(df.columns)}):\n{list(df.columns)}")
print(f"\nNull check:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nSanity checks:")
print(f"  callmoney:   {df['callmoney'].min():.2f} – {df['callmoney'].max():.2f}  (expect ~3-9)")
print(f"  usdinr ret:  {df['ret_t'].min():.4f} – {df['ret_t'].max():.4f}")
print(f"  rv range:    {df['rv'].min():.4f} – {df['rv'].max():.4f}")
print(f"  is_mpc_day:  {df['is_mpc_day'].sum()}  (expect ~60-80)")
print(f"  is_fomc_day: {df['is_fomc_day'].sum()}  (expect ~170)")
print(f"\nFirst date: {df.index.min().date()}, Last date: {df.index.max().date()}")

d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\testing\dl_projects\myenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\robin\AppData\Local\Temp\ipykernel_20892\549183651.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")


Saved: D:\MBFM_project\master_features.csv
Shape: (5199, 24)
Date range: 2005-04-04 → 2026-10-12

Columns (24):
['rv', 'rv_lag1', 'rv_5d', 'rv_22d', 'sq_return', 'vix_level', 'vix_chg', 'brent_ret', 'dxy_level', 'dxy_ret', 'india10y', 'india10y_chg', 'us10y', 'us10y_chg', 'rate_spread', 'callmoney', 'nifty_ret', 'em_ret', 'ret_t', 'is_mpc_day', 'is_fomc_day', 'day_of_week', 'month', 'week_of_year']

Null check:
Series([], dtype: int64)

Sanity checks:
  callmoney:   0.13 – 54.32  (expect ~3-9)
  usdinr ret:  -0.0301 – 0.0402
  rv range:    0.0041 – 0.2806
  is_mpc_day:  60  (expect ~60-80)
  is_fomc_day: 175  (expect ~170)

First date: 2005-04-04, Last date: 2026-10-12
